# Real Statements

This notebook explores personal finance concepts.


In [1]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# === Your First API Call ===
# The SEC requires a User-Agent header — it's how they know who's making requests.
# This is standard API etiquette. Use your name and email.
headers = {'User-Agent': 'YourName YourApp your.email@example.com'}

# Every public company has a CIK number — it's their SEC ID.
# Apple = 0000320193, Microsoft = 0000789019, Google (Alphabet) = 0001652044
companies = {
    'Apple':     '0000320193',
    'Microsoft': '0000789019',
    'Alphabet':  '0001652044',
}

# Let's pull Apple's financial data first
cik = companies['Apple']
url = f'https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json'

print(f"Calling: {url}")
print(f"This is a real HTTP request to the SEC's servers...\n")

response = requests.get(url, headers=headers)

print(f"Status code: {response.status_code}")
print(f"Response size: {len(response.text):,} characters")
print(f"Content type: {response.headers.get('Content-Type', 'unknown')}")

# Parse the JSON
data = response.json()

print(f"\nCompany: {data.get('entityName', 'Unknown')}")
print(f"CIK: {data.get('cik', 'Unknown')}")

# What financial concepts are available?
us_gaap = data.get('facts', {}).get('us-gaap', {})
print(f"Available financial metrics: {len(us_gaap)}")
print(f"\nFirst 15 metrics:")
for i, key in enumerate(list(us_gaap.keys())[:15]):
    print(f"  {i+1}. {key}")

Calling: https://data.sec.gov/api/xbrl/companyfacts/CIK0000320193.json
This is a real HTTP request to the SEC's servers...

Status code: 200
Response size: 3,709,629 characters
Content type: application/json

Company: Apple Inc.
CIK: 320193
Available financial metrics: 503

First 15 metrics:
  1. AccountsPayable
  2. AccountsPayableCurrent
  3. AccountsReceivableNetCurrent
  4. AccruedIncomeTaxesCurrent
  5. AccruedIncomeTaxesNoncurrent
  6. AccruedLiabilities
  7. AccruedLiabilitiesCurrent
  8. AccruedMarketingCostsCurrent
  9. AccumulatedDepreciationDepletionAndAmortizationPropertyPlantAndEquipment
  10. AccumulatedOtherComprehensiveIncomeLossAvailableForSaleSecuritiesAdjustmentNetOfTax
  11. AccumulatedOtherComprehensiveIncomeLossCumulativeChangesInNetGainLossFromCashFlowHedgesEffectNetOfTax
  12. AccumulatedOtherComprehensiveIncomeLossForeignCurrencyTranslationAdjustmentNetOfTax
  13. AccumulatedOtherComprehensiveIncomeLossNetOfTax
  14. AdjustmentsToAdditionalPaidInCapitalSharebas

In [2]:
# === Extract Real Financial Data from the SEC Response ===

def get_annual_data(facts, metric_name):
    """Pull annual (10-K) values for a given metric."""
    try:
        units = facts[metric_name]['units']
        # Financial values are usually in USD
        key = 'USD' if 'USD' in units else list(units.keys())[0]
        entries = units[key]
        
        # Filter for annual filings only (10-K), not quarterly (10-Q)
        annual = [e for e in entries if e.get('form') == '10-K']
        
        # Keep only the most recent entry per fiscal year
        by_year = {}
        for entry in annual:
            fy = entry.get('fy')
            if fy and fy >= 2019:  # Last 5 fiscal years
                by_year[fy] = entry['val']
        
        return by_year
    except (KeyError, IndexError):
        return {}

# The metrics we want — mapped to their XBRL names
metric_map = {
    'Revenue':           'Revenues',
    'Revenue_alt':       'RevenueFromContractWithCustomerExcludingAssessedTax',
    'Cost of Revenue':   'CostOfGoodsAndServicesSold',
    'Gross Profit':      'GrossProfit',
    'Operating Income':  'OperatingIncomeLoss',
    'Net Income':        'NetIncomeLoss',
    'Total Assets':      'Assets',
    'Total Liabilities': 'Liabilities',
    'Cash from Ops':     'NetCashProvidedByOperatingActivities',
    'CapEx':             'PaymentsToAcquirePropertyPlantAndEquipment',
}

# Extract Apple's data
apple_data = {}
for display_name, xbrl_name in metric_map.items():
    result = get_annual_data(us_gaap, xbrl_name)
    if result:
        apple_data[display_name] = result

# Build a clean dataframe
apple_df = pd.DataFrame(apple_data).T
apple_df.columns = [f'FY{yr}' for yr in apple_df.columns]
apple_df = apple_df.sort_index(axis=1)

# Convert to billions for readability
apple_display = (apple_df / 1e9).round(1)

print("APPLE INC. — Real Financial Data ($ Billions)")
print("=" * 60)
print(apple_display.to_string())
print(f"\nSource: SEC EDGAR (10-K annual filings)")

APPLE INC. — Real Financial Data ($ Billions)
                   FY2019  FY2020  FY2021  FY2022  FY2023  FY2024  FY2025
Revenue_alt          64.0    64.7   365.8   394.3   383.3   391.0   416.2
Cost of Revenue     161.8   169.6   213.0   223.5   214.1   210.4   221.0
Gross Profit         24.3    24.7   152.8   170.8   169.1   180.7   195.2
Operating Income     63.9    66.3   108.9   119.4   114.3   123.2   133.0
Net Income           13.7    12.7    94.7    99.8    97.0    93.7   112.0
Total Assets        338.5   323.9   351.0   352.8   352.6   365.0   359.2
Total Liabilities   248.0   258.5   287.9   302.1   290.4   308.0   285.5
CapEx                10.5     7.3    11.1    10.7    11.0     9.4    12.7

Source: SEC EDGAR (10-K annual filings)


In [3]:
# === Handle messy real-world data ===
# Try multiple XBRL names for the same concept — companies report differently

def get_metric(facts, names, min_year=2020):
    """Try multiple XBRL names until one returns data."""
    for name in names:
        result = get_annual_data(facts, name)
        # Filter to years we trust
        result = {k: v for k, v in result.items() if k >= min_year}
        if result:
            return result, name
    return {}, 'NOT FOUND'

# More robust metric mapping — multiple possible names for each concept
robust_metrics = {
    'Revenue': [
        'RevenueFromContractWithCustomerExcludingAssessedTax',
        'Revenues', 
        'SalesRevenueNet',
        'RevenueFromContractWithCustomerIncludingAssessedTax',
    ],
    'Cost of Revenue': [
        'CostOfGoodsAndServicesSold',
        'CostOfRevenue',
        'CostOfGoodsSold',
    ],
    'Gross Profit': ['GrossProfit'],
    'Operating Income': ['OperatingIncomeLoss'],
    'Net Income': ['NetIncomeLoss'],
    'Total Assets': ['Assets'],
    'Total Liabilities': ['Liabilities'],
    'Cash from Operations': [
        'NetCashProvidedByOperatingActivities',
        'CashProvidedByOperatingActivities',
    ],
    'Capital Expenditures': [
        'PaymentsToAcquirePropertyPlantAndEquipment',
        'CapitalExpenditure',
    ],
}

def pull_company_data(cik, company_name):
    """Pull all metrics for a company, handling messy data."""
    url = f'https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json'
    response = requests.get(url, headers=headers)
    
    if response.status_code != 200:
        print(f"ERROR: {company_name} returned status {response.status_code}")
        return None, {}
    
    facts = response.json().get('facts', {}).get('us-gaap', {})
    
    company_data = {}
    sources = {}
    for display_name, xbrl_names in robust_metrics.items():
        result, source = get_metric(facts, xbrl_names)
        if result:
            company_data[display_name] = result
            sources[display_name] = source
    
    return company_data, sources

# Pull all three companies
all_companies = {}
all_sources = {}

for name, cik in companies.items():
    print(f"Pulling {name}...", end=' ')
    data, sources = pull_company_data(cik, name)
    if data:
        all_companies[name] = data
        all_sources[name] = sources
        print(f"✓ Got {len(data)} metrics")
    else:
        print("✗ Failed")

# Show what XBRL names each company actually uses
print(f"\n{'Metric':<25} {'Apple':<45} {'Microsoft':<45} {'Alphabet':<45}")
print("-" * 160)
for metric in robust_metrics.keys():
    apple_src = all_sources.get('Apple', {}).get(metric, '—')
    msft_src = all_sources.get('Microsoft', {}).get(metric, '—')
    goog_src = all_sources.get('Alphabet', {}).get(metric, '—')
    print(f"{metric:<25} {apple_src:<45} {msft_src:<45} {goog_src:<45}")

Pulling Apple... ✓ Got 8 metrics
Pulling Microsoft... ✓ Got 8 metrics
Pulling Alphabet... ✓ Got 7 metrics

Metric                    Apple                                         Microsoft                                     Alphabet                                     
----------------------------------------------------------------------------------------------------------------------------------------------------------------
Revenue                   RevenueFromContractWithCustomerExcludingAssessedTax RevenueFromContractWithCustomerExcludingAssessedTax RevenueFromContractWithCustomerExcludingAssessedTax
Cost of Revenue           CostOfGoodsAndServicesSold                    CostOfGoodsAndServicesSold                    CostOfRevenue                                
Gross Profit              GrossProfit                                   GrossProfit                                   —                                            
Operating Income          OperatingIncomeLoss             

In [4]:
# === Find the right name for Cash from Operations ===
# Let's search Apple's available metrics for anything with "Cash" in the name

cash_metrics = [key for key in us_gaap.keys() if 'Cash' in key or 'cash' in key]
print(f"Apple metrics containing 'Cash' ({len(cash_metrics)}):\n")
for m in sorted(cash_metrics):
    # Show the most recent value so we can identify the right one
    try:
        entries = us_gaap[m]['units']
        key = 'USD' if 'USD' in entries else list(entries.keys())[0]
        vals = [e for e in entries[key] if e.get('form') == '10-K' and e.get('fy', 0) >= 2024]
        if vals:
            latest = vals[-1]['val']
            print(f"  {m:<65} FY{vals[-1]['fy']}: ${latest/1e9:.1f}B")
    except:
        pass

Apple metrics containing 'Cash' (27):

  CashAndCashEquivalentsAtCarryingValue                             FY2025: $35.9B
  CashCashEquivalentsRestrictedCashAndRestrictedCashEquivalents     FY2025: $35.9B
  CashCashEquivalentsRestrictedCashAndRestrictedCashEquivalentsPeriodIncreaseDecreaseIncludingExchangeRateEffect FY2025: $6.0B
  NetCashProvidedByUsedInFinancingActivities                        FY2025: $-120.7B
  NetCashProvidedByUsedInInvestingActivities                        FY2025: $15.2B
  NetCashProvidedByUsedInOperatingActivities                        FY2025: $111.5B
  OtherComprehensiveIncomeLossCashFlowHedgeGainLossAfterReclassificationAndTax FY2025: $0.6B
  OtherComprehensiveIncomeLossCashFlowHedgeGainLossBeforeReclassificationAfterTax FY2025: $0.8B
  OtherComprehensiveIncomeLossCashFlowHedgeGainLossReclassificationAfterTax FY2025: $0.2B
  OtherNoncashIncomeExpense                                         FY2025: $0.1B


In [5]:
# === Updated metrics with correct cash flow names ===
robust_metrics['Cash from Operations'] = [
    'NetCashProvidedByUsedInOperatingActivities',
    'NetCashProvidedByOperatingActivities',
]

# Re-pull all three companies with the fix
all_companies = {}
for name, cik in companies.items():
    data, sources = pull_company_data(cik, name)
    all_companies[name] = data

# Build comparison table for most recent fiscal year
# Find common years across all companies
common_years = None
for company_data in all_companies.values():
    for metric_data in company_data.values():
        years = set(metric_data.keys())
        common_years = years if common_years is None else common_years & years

latest_year = max(common_years) if common_years else 2024
print(f"Comparing fiscal year: FY{latest_year}\n")

# Build the comparison
comparison = {}
metrics_to_show = ['Revenue', 'Cost of Revenue', 'Gross Profit', 'Operating Income', 
                   'Net Income', 'Cash from Operations', 'Capital Expenditures',
                   'Total Assets', 'Total Liabilities']

for metric in metrics_to_show:
    row = {}
    for company in companies.keys():
        val = all_companies.get(company, {}).get(metric, {}).get(latest_year)
        row[company] = val
    comparison[metric] = row

comp_df = pd.DataFrame(comparison).T
comp_df = (comp_df / 1e9).round(1)

print(f"{'Metric':<25} {'Apple':>10} {'Microsoft':>10} {'Alphabet':>10}")
print("-" * 58)
for metric in metrics_to_show:
    vals = []
    for company in ['Apple', 'Microsoft', 'Alphabet']:
        v = comp_df.loc[metric, company] if metric in comp_df.index else None
        vals.append(f"${v:.1f}B" if pd.notna(v) else "N/A")
    print(f"{metric:<25} {vals[0]:>10} {vals[1]:>10} {vals[2]:>10}")

# Calculate key ratios
print(f"\n{'--- KEY RATIOS ---':^58}")
print(f"{'Ratio':<25} {'Apple':>10} {'Microsoft':>10} {'Alphabet':>10}")
print("-" * 58)

for company in ['Apple', 'Microsoft', 'Alphabet']:
    rev = all_companies[company].get('Revenue', {}).get(latest_year)
    gp = all_companies[company].get('Gross Profit', {}).get(latest_year)
    oi = all_companies[company].get('Operating Income', {}).get(latest_year)
    ni = all_companies[company].get('Net Income', {}).get(latest_year)
    
    if rev and gp:
        comp_df.loc['Gross Margin', company] = round(gp/rev*100, 1)
    if rev and oi:
        comp_df.loc['Operating Margin', company] = round(oi/rev*100, 1)
    if rev and ni:
        comp_df.loc['Net Margin', company] = round(ni/rev*100, 1)

for ratio in ['Gross Margin', 'Operating Margin', 'Net Margin']:
    vals = []
    for company in ['Apple', 'Microsoft', 'Alphabet']:
        v = comp_df.loc[ratio, company] if ratio in comp_df.index else None
        vals.append(f"{v:.1f}%" if pd.notna(v) else "N/A")
    print(f"{ratio:<25} {vals[0]:>10} {vals[1]:>10} {vals[2]:>10}")

Comparing fiscal year: FY2024

Metric                         Apple  Microsoft   Alphabet
----------------------------------------------------------
Revenue                      $391.0B    $245.1B    $350.0B
Cost of Revenue              $210.4B     $74.1B    $146.3B
Gross Profit                 $180.7B    $171.0B        N/A
Operating Income             $123.2B    $109.4B    $112.4B
Net Income                    $93.7B     $88.1B    $100.1B
Cash from Operations         $118.3B    $118.5B    $125.3B
Capital Expenditures           $9.4B     $44.5B     $52.5B
Total Assets                 $365.0B    $512.2B    $450.3B
Total Liabilities            $308.0B    $243.7B    $125.2B

                    --- KEY RATIOS ---                    
Ratio                          Apple  Microsoft   Alphabet
----------------------------------------------------------
Gross Margin                   46.2%      69.8%        N/A
Operating Margin               31.5%      44.6%      32.1%
Net Margin              

## Try This

- The code fetches real SEC data. Try a different company CIK if you know one.
- Experiment with the years or metrics pulled.
- Note how the parsing handles different company naming conventions.
